<a href="https://colab.research.google.com/github/Aaguilar123/Macro-Liquidity-Asset-Leaderboard-Forecast/blob/Aaguilar123-patch-1/shared_eda/Data_pull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Collection and Preparation Pipeline

##Environment Setup and Parameter Configuration

In [5]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web
from datetime import datetime

# 1. Project Parameters
START_DATE = '2015-01-01'
END_DATE = '2026-04-15'

# The 5 Market Assets
YAHOO_TICKERS = ['QQQ', 'XLP', 'GLD', 'BTC-USD', 'UUP']

# The 3 Core Macro Indicators:
FRED_TICKERS = ['FEDFUNDS', 'CPIAUCSL', 'UNRATE']

##Market Data Collection and Processing

In [6]:
def pull_market_data():
    print(f"Pulling Market Data from Yahoo Finance to calculate Dollar Volume...")
    data = yf.download(YAHOO_TICKERS, start=START_DATE, end=END_DATE)
    prices = data['Close'].ffill()
    volume = data['Volume'].ffill()


    dollar_volume = volume.copy()
    # ETFs are in shares -> Multiply by Price
    etfs = ['QQQ', 'XLP', 'GLD', 'UUP']
    for etf in etfs:
        dollar_volume[etf] = prices[etf] * volume[etf]

    # Bitcoin is ALREADY in Dollars
    dollar_volume['BTC-USD'] = volume['BTC-USD']

    return dollar_volume

##Macroeconomic Data Collection


In [7]:
def pull_macro_data():
    print(f"Pulling Macro Data from FRED ({START_DATE} to {END_DATE})...")
    # FRED API
    macro_data = web.DataReader(FRED_TICKERS, 'fred', START_DATE, END_DATE)

    macro_data.rename(columns={
        'FEDFUNDS': 'Fed_Rate',
        'CPIAUCSL': 'CPI_Inflation',
        'UNRATE': 'Unemployment_Rate'
    }, inplace=True)

    # Force the index name to match Yahoo's exactly
    macro_data.index.name = 'Date'

    return macro_data

##Data Execution and Export

In [8]:
if __name__ == "__main__":
    df_volume = pull_market_data()
    df_macro = pull_macro_data()

    print("\nMarket Volume Data Preview:")
    print(df_volume.head())

    print("\nMacro Data Preview:")
    print(df_macro.head())

    # Export to CSV
    print("\nExporting data to CSV files...")
    df_volume.to_csv('market_volume.csv')
    df_macro.to_csv('macro_indicators.csv')

    print("The 2 CSV files are ready.")

/tmp/ipykernel_2782/1375974141.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(YAHOO_TICKERS, start=START_DATE, end=END_DATE)
[*******************   40%                       ]  2 of 5 completed

Pulling Market Data from Yahoo Finance to calculate Dollar Volume...


[*********************100%***********************]  5 of 5 completed


Pulling Macro Data from FRED (2015-01-01 to 2026-04-15)...

Market Volume Data Preview:
Ticker       BTC-USD           GLD           QQQ           UUP           XLP
Date                                                                        
2015-01-01   8036550           NaN           NaN           NaN           NaN
2015-01-02   7860650  8.110632e+08  2.964399e+09  3.816104e+07  2.730110e+08
2015-01-03  33054400  8.110632e+08  2.964399e+09  3.816104e+07  2.730110e+08
2015-01-04  55629100  8.110632e+08  2.964399e+09  3.816104e+07  2.730110e+08
2015-01-05  43962800  9.469429e+08  3.406576e+09  5.784805e+07  2.946172e+08

Macro Data Preview:
            Fed_Rate  CPI_Inflation  Unemployment_Rate
Date                                                  
2015-01-01      0.11        234.747                5.7
2015-02-01      0.11        235.342                5.5
2015-03-01      0.11        235.976                5.4
2015-04-01      0.12        236.222                5.4
2015-05-01      0.12  